<a href="https://colab.research.google.com/github/abhishek18-blog/DataScience-and-ML/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain_community langchainhub chromadb longchain langchain-openai

In [ ]:
!pip install langchain_groq

In [ ]:
import os
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq

# 1. Set your NEW Groq API Key
os.environ["GROQ_API_KEY"] = "your_api_key"

print("1. Loading website data...")
# Note: Because of Cloudflare, this might still just scrape "Checking user..."
# To bypass Cloudflare properly, refer to the Playwright workaround provided earlier.
url = "https://educosys.com/course/details?course-id=fa7330e0-d2c0-4d77-902d-2834a7984f8d"
loader = WebBaseLoader(web_path=[url])
docs = loader.load()

print("2. Splitting text...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
splits = text_splitter.split_documents(docs)
print(f"Created {len(splits)} splits.")

print("3. Creating Vector Database with HuggingFace (This downloads the model on the first run)...")
# Note: The "UNEXPECTED" BertModel warning in the console is completely normal for this library, safely ignore it!
hf_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=splits, embedding=hf_embeddings)

print("4. Initializing Groq AI...")
llm = ChatGroq(
    temperature=0,
model_name="llama-3.1-8b-instant" # Updated to the newest available model)
)
print("5. Testing Groq...")
# Using the proper message format [("human", "message")] prevents BadRequestErrors
try:
    response = llm.invoke([("human", "Hi Groq is certificate available in the course?")])
    print("\n✅ SUCCESS! Groq says:")
    print(response.content)
except Exception as e:
    print(f"\n❌ ERROR: {e}")
    print("If you get an error here, your API key might be invalid or not active yet.")

In [ ]:
    response = llm.invoke([("human", "Hi Groq is certificate available in the course?")])
    print(response.content)
